In [0]:
%pip install --quiet h2o interpret h2o_pysparkling_3.5
#%%sh pip install --quiet h2o

In [0]:
#dbutils.library.restartPython()

In [0]:
import h2o
from h2o.automl import H2OAutoML
from pysparkling import H2OContext
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import pickle
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pysparkling import H2OContext
from sklearn.impute import KNNImputer
from h2o.automl import H2OAutoML

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [0]:
model_train_data_final = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_trainingdata', header = True)
test_data_final = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_testdata', header = True)

#model_train_data_final_pd = model_train_data_final.toPandas()
#test_data_final_pd = test_data_final.toPandas()

model_train_data_final.display()
test_data_final.display()

In [0]:
# Ordinal Encoding for Segmentations

def ordinal_transform(df):
    # Use string representations for replacement values
    seg_mapping = {"Unassigned": "0", "L": "1", "M": "2", "H": "3"}
    dig_mapping = {"New": "0", "Unassigned": "1", "L": "2", "M": "3", "H": "4"}

    seg_columns = ["price_dim_seg", "health_dim_seg", "convenience_dim_seg", "variety_seeking_dim_seg"]
    all_target_cols = seg_columns + ["dig_eng_seg_code"]

    for col in all_target_cols:
        df = df.withColumn(col, f.trim(f.col(col)))

    df = df.replace(to_replace=seg_mapping, subset=seg_columns)
    df = df.replace(to_replace=dig_mapping, subset=["dig_eng_seg_code"])
    
    for col in all_target_cols:
        df = df.withColumn(col, f.col(col).cast("integer"))

    return df

In [0]:
model_train_data_final = ordinal_transform(model_train_data_final)
test_data_final = ordinal_transform(test_data_final)

model_train_data_final.display()

In [0]:
# Create H2O instance 
h2o.init() #nthreads=4, max_mem_size="4G"

In [0]:
# Initialize H2O Context
hc = H2OContext.getOrCreate()

# Convert PySpark DataFrames to H20Frame
h2o_train = hc.asH2OFrame(model_train_data_final)
h2o_test = hc.asH2OFrame(test_data_final)

In [0]:
# Run AutoML excluding Target Label and EHHN
target_col = "target_label" 
id_col = "ehhn"

h2o_train[target_col] = h2o_train[target_col].asfactor()
feature_cols = [col for col in h2o_train.columns if col not in [target_col, id_col]]

auto_ml = H2OAutoML(
    max_runtime_secs=600, 
    max_models=40, 
    seed=8451, 
    nfolds=4, 
    exclude_algos=['StackedEnsemble']
)

# 5. Train with x, y, and training_frame explicitly passed
auto_ml.train(
    x=feature_cols, 
    y=target_col, 
    training_frame=h2o_train
)

In [0]:
# View leaderboard
leaderboard = auto_ml.leaderboard
leaderboard.head()

#